<a href="https://colab.research.google.com/github/itz-naman007/Dense-Encoder-Fine-Tuning-Wiki-QA-/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import torch
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator

In [28]:
dataset = load_dataset("microsoft/wiki_qa", split="train")
# MNRL expects datasets with "anchor" (query) and "positive" (document) columns
dataset = dataset.rename_columns({"question": "anchor", "answer": "positive"})
# Drop extraneous metadata columns
dataset = dataset.select_columns(["anchor", "positive"])
# Shuffle to ensure diverse contexts, then slice 500 for training and 100 for evaluation
dataset = dataset.shuffle(seed=42)
train_dataset = dataset.select(range(500))
eval_dataset = dataset.select(range(500, 600))

In [29]:
from datasets import Dataset

# 2. Convert to Pandas to deduplicate
df = dataset.to_pandas()

# CRITICAL FIX: Drop duplicate documents so each context appears exactly once
df = df.drop_duplicates(subset=["positive"]).reset_index(drop=True)

# 3. Convert back to Hugging Face Dataset
deduped_dataset = Dataset.from_pandas(df)

In [30]:
# 2. Extract Evaluation Lists
eval_queries = eval_dataset["anchor"]
eval_docs = eval_dataset["positive"]

In [31]:
# 3. Initialize Base Model and Loss Function
model_id = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(model_id)
loss = MultipleNegativesRankingLoss(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [32]:
# 4. Set Up Information Retrieval Evaluator
# Map indices to string IDs for the evaluator
corpus = {str(i): doc for i, doc in enumerate(eval_docs)}
queries_dict = {str(i): query for i, query in enumerate(eval_queries)}

# Assuming a 1-to-1 relationship for this evaluation slice
relevant_docs = {str(i): {str(i)} for i in range(len(eval_queries))}

ir_evaluator = InformationRetrievalEvaluator(
    queries=queries_dict,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="qa-eval"
)

# Run Pre-Fine-Tune Evaluation
pre_eval_results = ir_evaluator(model)
pre_recall_5 = pre_eval_results.get('qa-eval_cosine_recall@5', 0.0)

In [33]:
# 5. Configure Training Arguments
args = SentenceTransformerTrainingArguments(
    output_dir="./domain_encoder",
    num_train_epochs=3,
    per_device_train_batch_size=32, # Increase to 32 or 64 if VRAM allows
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
)

In [34]:
# 6. Initialize Trainer and Fine-Tune
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=ir_evaluator,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,Qa-eval Cosine Accuracy@1,Qa-eval Cosine Accuracy@3,Qa-eval Cosine Accuracy@5,Qa-eval Cosine Accuracy@10,Qa-eval Cosine Precision@1,Qa-eval Cosine Precision@3,Qa-eval Cosine Precision@5,Qa-eval Cosine Precision@10,Qa-eval Cosine Recall@1,Qa-eval Cosine Recall@3,Qa-eval Cosine Recall@5,Qa-eval Cosine Recall@10,Qa-eval Cosine Ndcg@10,Qa-eval Cosine Mrr@10,Qa-eval Cosine Map@100
1,1.151335,No log,0.650000,0.800000,0.860000,0.930000,0.650000,0.266667,0.172000,0.093000,0.650000,0.800000,0.860000,0.930000,0.793091,0.749135,0.754026
2,0.698978,No log,0.700000,0.820000,0.890000,0.940000,0.700000,0.273333,0.178000,0.094000,0.700000,0.820000,0.890000,0.940000,0.816483,0.777357,0.781929
3,0.566912,No log,0.700000,0.810000,0.890000,0.940000,0.700000,0.270000,0.178000,0.094000,0.700000,0.810000,0.890000,0.940000,0.816526,0.777313,0.781854


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=48, training_loss=0.7726975679397583, metrics={'train_runtime': 12.0184, 'train_samples_per_second': 124.809, 'train_steps_per_second': 3.994, 'total_flos': 0.0, 'train_loss': 0.7726975679397583, 'epoch': 3.0})

In [36]:
# 7. Run Post-Fine-Tune Evaluation
post_eval_results = ir_evaluator(model)
post_recall_5 = post_eval_results.get('qa-eval_cosine_recall@5', 0.0)

print(f"Pre-Fine-Tune Recall@5:  {pre_recall_5:.4f}")
print(f"Post-Fine-Tune Recall@5: {post_recall_5:.4f}")

model.save_pretrained("./domain_encoder_final")

Pre-Fine-Tune Recall@5:  0.7900
Post-Fine-Tune Recall@5: 0.8900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [38]:
from huggingface_hub import login
from sentence_transformers import SentenceTransformer

# Log in programmatically (it will prompt you for your token)
login()

# Load and push the model
model = SentenceTransformer("./domain_encoder_final")
model.push_to_hub("nmngpt0/bge-qa-lilsmall01")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'https://huggingface.co/nmngpt0/bge-qa-lilsmall01/commit/9beb3a29e5ee65cf97400d287138eb2f02899b48'